# Intermediate Object-Oriented Programming in Python

## Table of Contents
1. [Abstract Classes (ABC, @abstractmethod)](#abstract-classes)
2. [Interfaces in Python](#interfaces)
3. [Composition vs Aggregation](#composition-vs-aggregation)
4. [@property Decorator](#property-decorator)
5. [Name Mangling](#name-mangling)
6. [__slots__](#slots)
7. [Magic Methods](#magic-methods)
8. [Named Tuples vs Data Classes](#named-tuples-vs-data-classes)
9. [Nested Classes](#nested-classes)
10. [LeetCode Problems](#leetcode-problems)

## 1. Abstract Classes (ABC, @abstractmethod) <a id="abstract-classes"></a>

An **abstract class** is a class that cannot be instantiated directly and is meant to be subclassed.

### Key Points:
- Use the `ABC` (Abstract Base Class) module from `abc`
- Abstract methods are decorated with `@abstractmethod`
- Subclasses must implement all abstract methods
- Abstract classes define a **contract** for subclasses

### Example:

In [ ]:
from abc import ABC, abstractmethod

class Shape(ABC):
    """Abstract base class for all shapes."""
    
    def __init__(self, color):
        self.color = color
    
    @abstractmethod
    def area(self):
        """Calculate the area of the shape."""
        pass
    
    @abstractmethod
    def perimeter(self):
        """Calculate the perimeter of the shape."""
        pass
    
    def describe(self):
        """Concrete method that can be used by all subclasses."""
        return f"A {self.color} shape"

# This would raise an error:
# shape = Shape('red')  # TypeError: Can't instantiate abstract class

class Rectangle(Shape):
    def __init__(self, color, width, height):
        super().__init__(color)
        self.width = width
        self.height = height
    
    def area(self):
        return self.width * self.height
    
    def perimeter(self):
        return 2 * (self.width + self.height)

class Circle(Shape):
    def __init__(self, color, radius):
        super().__init__(color)
        self.radius = radius
    
    def area(self):
        import math
        return math.pi * self.radius ** 2
    
    def perimeter(self):
        import math
        return 2 * math.pi * self.radius

rect = Rectangle('red', 5, 3)
circle = Circle('blue', 4)

print(f"Rectangle: {rect.describe()}, Area: {rect.area()}, Perimeter: {rect.perimeter()}")
print(f"Circle: {circle.describe()}, Area: {circle.area():.2f}, Perimeter: {circle.perimeter():.2f}")

## 2. Interfaces in Python <a id="interfaces"></a>

Python doesn't have a built-in `interface` keyword like Java, but we can simulate interfaces using abstract classes.

### Interface vs Abstract Class:
| Feature | Interface | Abstract Class |
|---------|-----------|----------------|
| Methods | All abstract | Can have concrete methods |
| Variables | All class constants | Can have instance variables |
| Inheritance | Multiple inheritance | Single inheritance (typically) |
| Purpose | Define a contract | Provide a base implementation |

### Example:

In [ ]:
from abc import ABC, abstractmethod

class Drawable(ABC):
    """Interface for drawable objects."""
    
    @abstractmethod
    def draw(self):
        pass
    
    @abstractmethod
    def get_area(self):
        pass

class Resizable(ABC):
    """Interface for resizable objects."""
    
    @abstractmethod
    def resize(self, factor):
        pass

class Square(Drawable, Resizable):
    def __init__(self, side):
        self.side = side
    
    def draw(self):
        return f"Drawing a square with side {self.side}"
    
    def get_area(self):
        return self.side ** 2
    
    def resize(self, factor):
        self.side *= factor
        return f"Resized square to side {self.side}"

square = Square(5)
print(square.draw())
print(f"Area: {square.get_area()}")
print(square.resize(2))
print(f"New area: {square.get_area()}")

# Check if it implements the interfaces
print(f"Is Drawable: {isinstance(square, Drawable)}")
print(f"Is Resizable: {isinstance(square, Resizable)}")

## 3. Composition vs Aggregation <a id="composition-vs-aggregation"></a>

Both composition and aggregation are forms of **object containment** (has-a relationship).

### Composition (Strong Ownership):
- The child cannot exist without the parent
- If the parent is destroyed, the child is destroyed
- Uses the `__init__` method to create child objects

### Aggregation (Weak Ownership):
- The child can exist independently of the parent
- If the parent is destroyed, the child can still exist
- Child objects are passed in or created externally

### Example:

In [ ]:
# Composition: Engine cannot exist without Car
class Engine:
    def __init__(self, horsepower):
        self.horsepower = horsepower
    
    def start(self):
        return f"Engine with {self.horsepower} HP started"

class Car:
    def __init__(self, make, horsepower):
        self.make = make
        # Composition: Engine is created inside Car
        self.engine = Engine(horsepower)
    
    def start(self):
        return f"{self.make} car: {self.engine.start()}"

# Aggregation: Student can exist without University
class Student:
    def __init__(self, name):
        self.name = name
    
    def get_info(self):
        return f"Student: {self.name}"

class University:
    def __init__(self, name):
        self.name = name
        # Aggregation: Students are passed in, not created here
        self.students = []
    
    def add_student(self, student):
        self.students.append(student)
    
    def get_students(self):
        return [s.get_info() for s in self.students]

# Using composition
car = Car('Tesla', 500)
print(car.start())

# Using aggregation
student1 = Student('Alice')
student2 = Student('Bob')
uni = University('MIT')
uni.add_student(student1)
uni.add_student(student2)
print(f"University: {uni.name}")
print(f"Students: {uni.get_students()}")
print(f"Student1 still exists: {student1.get_info()}")

## 4. @property Decorator <a id="property-decorator"></a>

The `@property` decorator provides a Pythonic way to create getters, setters, and deleters.

### Key Points:
- `@property` creates a getter method
- `@<property_name>.setter` creates a setter method
- `@<property_name>.deleter` creates a deleter method
- Allows you to add validation while maintaining attribute-like access

### Example:

In [ ]:
class Temperature:
    def __init__(self, celsius=0):
        self._celsius = celsius
    
    @property
    def celsius(self):
        """Getter for celsius temperature."""
        return self._celsius
    
    @celsius.setter
    def celsius(self, value):
        """Setter for celsius temperature with validation."""
        if value < -273.15:
            raise ValueError("Temperature cannot be below absolute zero!")
        self._celsius = value
    
    @property
    def fahrenheit(self):
        """Computed property: Fahrenheit from Celsius."""
        return self._celsius * 9/5 + 32
    
    @fahrenheit.setter
    def fahrenheit(self, value):
        """Setter for fahrenheit that updates celsius."""
        self.celsius = (value - 32) * 5/9
    
    @celsius.deleter
    def celsius(self):
        """Deleter for celsius temperature."""
        del self._celsius

temp = Temperature(25)
print(f"Celsius: {temp.celsius}")
print(f"Fahrenheit: {temp.fahrenheit}")

temp.fahrenheit = 100
print(f"After setting 100F -> Celsius: {temp.celsius:.2f}, Fahrenheit: {temp.fahrenheit}")

try:
    temp.celsius = -300
except ValueError as e:
    print(f"Error: {e}")

del temp.celsius
print(f"After deletion: {hasattr(temp, '_celsius')}")

## 5. Name Mangling <a id="name-mangling"></a>

Name mangling is Python's mechanism for avoiding name collisions in subclasses.

### Key Points:
- Any identifier with two leading underscores and at most one trailing underscore is textually replaced with `_classname__identifier`
- This prevents accidental overriding of private attributes
- It's not a security feature - it's a convention to avoid name conflicts

### Example:

In [ ]:
class MyClass:
    def __init__(self):
        self.public = "public"
        self._protected = "protected"
        self.__private = "private"
    
    def get_private(self):
        return self.__private

class SubClass(MyClass):
    def __init__(self):
        super().__init__()
        self.__private = "subclass private"
    
    def show_mangling(self):
        # These are different attributes due to name mangling
        print(f"Parent's private: {self._MyClass__private}")
        print(f"Subclass's private: {self._SubClass__private}")

obj = MyClass()
print(f"Public: {obj.public}")
print(f"Protected: {obj._protected}")
print(f"Private (via name mangling): {obj._MyClass__private}")
print(f"Private (via method): {obj.get_private()}")

sub = SubClass()
sub.show_mangling()
print(f"Public (from subclass): {sub.public}")

## 6. __slots__ <a id="slots"></a>

By default, Python stores object attributes in a `__dict__` dictionary. The `__slots__` attribute allows you to specify exactly which attributes an object can have.

### Benefits:
- **Memory savings**: No `__dict__` overhead
- **Faster attribute access**: Direct memory access instead of dictionary lookup
- **Prevents typos**: Can't add unexpected attributes

### Limitations:
- Can't add new attributes not defined in `__slots__`
- Doesn't support multiple inheritance with classes that have `__dict__`

### Example:

In [ ]:
class WithoutSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class WithSlots:
    __slots__ = ['x', 'y']  # Only these attributes are allowed
    
    def __init__(self, x, y):
        self.x = x
        self.y = y

obj1 = WithoutSlots(1, 2)
obj2 = WithSlots(1, 2)

print(f"Without __slots__ - has __dict__: {hasattr(obj1, '__dict__')}")
print(f"With __slots__ - has __dict__: {hasattr(obj2, '__dict__')}")

# With __slots__, can't add new attributes
try:
    obj2.z = 3
except AttributeError as e:
    print(f"Error: {e}")

# Without __slots__, can add new attributes
obj1.z = 3
print(f"Without __slots__, added z: {obj1.z}")

# Memory comparison
import sys
obj1 = WithoutSlots(1, 2)
obj2 = WithSlots(1, 2)
print(f"\nMemory without __slots__: {sys.getsizeof(obj1)} bytes")
print(f"Memory with __slots__: {sys.getsizeof(obj2)} bytes")
print(f"Memory with __slots__ + __dict__: {sys.getsizeof(obj2) + sys.getsizeof(obj2.__dict__ if hasattr(obj2, '__dict__') else {})} bytes")

## 7. Magic Methods <a id="magic-methods"></a>

Magic methods (dunder methods) are special methods that enable Python's built-in behavior.

### Common Magic Methods:

| Method | Description | Triggered by |
|--------|-------------|--------------|
| `__new__` | Object creation | `ClassName()` |
| `__init__` | Object initialization | `ClassName()` |
| `__call__` | Make objects callable | `obj()` |
| `__len__` | Length | `len(obj)` |
| `__getitem__` | Index access | `obj[key]` |
| `__setitem__` | Index assignment | `obj[key] = val` |
| `__delitem__` | Index deletion | `del obj[key]` |
| `__iter__` | Iterator creation | `for x in obj` |
| `__contains__` | Membership test | `x in obj` |
| `__enter__` | Context manager entry | `with obj as x` |
| `__exit__` | Context manager exit | `with obj as x` |

### Example:

In [ ]:
class MyContainer:
    def __init__(self, items=None):
        self._items = items if items is not None else []
    
    def __new__(cls, *args, **kwargs):
        """Called before __init__ when creating a new instance."""
        instance = super().__new__(cls)
        return instance
    
    def __len__(self):
        """Return the number of items."""
        return len(self._items)
    
    def __getitem__(self, index):
        """Get item by index."""
        return self._items[index]
    
    def __setitem__(self, index, value):
        """Set item by index."""
        self._items[index] = value
    
    def __delitem__(self, index):
        """Delete item by index."""
        del self._items[index]
    
    def __iter__(self):
        """Return an iterator."""
        return iter(self._items)
    
    def __contains__(self, item):
        """Check if item is in container."""
        return item in self._items
    
    def __call__(self, func):
        """Make the container callable as a decorator."""
        def wrapper(*args, **kwargs):
            print(f"Before calling {func.__name__}")
            result = func(*args, **kwargs)
            print(f"After calling {func.__name__}")
            return result
        return wrapper

container = MyContainer([1, 2, 3, 4, 5])

print(f"Length: {len(container)}")
print(f"Item at index 2: {container[2]}")
container[0] = 10
print(f"After setting index 0: {container[0]}")
print(f"Contains 3: {3 in container}")
print(f"Iterating: {list(container)}")

# Using __call__ as a decorator
@container
def greet(name):
    return f"Hello, {name}!"

print(greet('Alice'))

## 8. Named Tuples vs Data Classes <a id="named-tuples-vs-data-classes"></a>

Both are used to create lightweight, immutable data containers.

### Named Tuples:
- From the `collections` module
- Immutable (cannot change values after creation)
- Memory efficient
- Can be unpacked like regular tuples
- Don't support default values (before Python 3.7)

### Data Classes:
- From the `dataclasses` module (Python 3.7+)
- Mutable by default
- Support default values, type hints, and validation
- Auto-generate `__init__`, `__repr__`, `__eq__`, etc.
- Can be frozen (immutable) with `@dataclass(frozen=True)`

### Example:

In [ ]:
from collections import namedtuple
from dataclasses import dataclass, field
from typing import List

# Named Tuple
Point = namedtuple('Point', ['x', 'y'])
p1 = Point(3, 4)
print(f"NamedTuple: {p1}")
print(f"x: {p1.x}, y: {p1.y}")
print(f"Unpacked: {p1[0]}, {p1[1]}")

# Data Class
@dataclass
class PointDC:
    x: float
    y: float
    
    def distance_from_origin(self):
        return (self.x**2 + self.y**2)**0.5

p2 = PointDC(3, 4)
print(f"\nDataClass: {p2}")
print(f"x: {p2.x}, y: {p2.y}")
print(f"Distance from origin: {p2.distance_from_origin()}")

# Data Class with defaults
@dataclass
class Person:
    name: str
    age: int = 0
    hobbies: List[str] = field(default_factory=list)

person1 = Person('Alice', 30)
person2 = Person('Bob')
person1.hobbies.append('Reading')

print(f"\nPerson 1: {person1}")
print(f"Person 2: {person2}")
print(f"Person 1 hobbies: {person1.hobbies}")
print(f"Person 2 hobbies: {person2.hobbies}")  # Different list!

# Frozen Data Class (immutable)
@dataclass(frozen=True)
class ImmutablePoint:
    x: float
    y: float

ip = ImmutablePoint(3, 4)
print(f"\nFrozen DataClass: {ip}")
try:
    ip.x = 5
except Exception as e:
    print(f"Cannot modify: {type(e).__name__}")

## 9. Nested Classes <a id="nested-classes"></a>

A nested class is a class defined inside another class.

### Key Points:
- The outer class can contain multiple inner classes
- Inner classes can access the outer class's members
- Inner classes are just namespaced within the outer class
- Useful for grouping related classes together

### Example:

In [ ]:
class LinkedList:
    """A linked list implementation with a nested Node class."""
    
    class Node:
        """Inner class representing a node in the linked list."""
        def __init__(self, data):
            self.data = data
            self.next = None
        
        def __str__(self):
            return str(self.data)
    
    def __init__(self):
        self.head = None
        self.size = 0
    
    def append(self, data):
        """Add a node to the end of the list."""
        new_node = self.Node(data)
        if not self.head:
            self.head = new_node
        else:
            current = self.head
            while current.next:
                current = current.next
            current.next = new_node
        self.size += 1
    
    def __iter__(self):
        """Make the list iterable."""
        current = self.head
        while current:
            yield current.data
            current = current.next
    
    def __len__(self):
        return self.size
    
    def __str__(self):
        values = list(self)
        return ' -> '.join(str(v) for v in values) + ' -> None'

# Using the linked list
ll = LinkedList()
ll.append(1)
ll.append(2)
ll.append(3)

print(f"Linked List: {ll}")
print(f"Length: {len(ll)}")
print(f"Iterating: {list(ll)}")

# Accessing the inner class
Node = LinkedList.Node
node = Node(42)
print(f"Standalone node: {node}")

# Another example: Builder pattern with nested classes
class Computer:
    def __init__(self):
        self.cpu = None
        self.ram = None
        self.storage = None
    
    class Builder:
        """Builder pattern using a nested class."""
        def __init__(self):
            self._computer = Computer()
        
        def set_cpu(self, cpu):
            self._computer.cpu = cpu
            return self
        
        def set_ram(self, ram):
            self._computer.ram = ram
            return self
        
        def set_storage(self, storage):
            self._computer.storage = storage
            return self
        
        def build(self):
            return self._computer
    
    def __str__(self):
        return f"Computer(CPU: {self.cpu}, RAM: {self.ram}, Storage: {self.storage})"

computer = Computer.Builder().set_cpu('i9').set_ram('32GB').set_storage('1TB SSD').build()
print(f"\n{computer}")

## 10. LeetCode Problems <a id="leetcode-problems"></a>

Now let's apply intermediate OOP concepts to solve some LeetCode problems.

### Problem 1: LRU Cache (LeetCode #146)

Design a data structure that follows the constraints of a Least Recently Used (LRU) cache.

Implement the `LRUCache` class:
- `int get(int key)` Return the value of the key if it exists, otherwise -1.
- `void put(int key, int value)` Update the value of the key if it exists, otherwise add the key-value pair.

If the number of keys exceeds the capacity, evict the least recently used key.

The functions `get` and `put` must each run in O(1) average time complexity.

In [ ]:
class LRUCache:
    class _Node:
        """Doubly linked list node for LRU cache."""
        __slots__ = ('key', 'value', 'prev', 'next')
        
        def __init__(self, key=0, value=0):
            self.key = key
            self.value = value
            self.prev = None
            self.next = None
    
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}  # key -> Node
        # Sentinel nodes for doubly linked list
        self._head = self._Node()  # Most recently used
        self._tail = self._Node()  # Least recently used
        self._head.next = self._tail
        self._tail.prev = self._head
    
    def _remove(self, node):
        """Remove a node from the linked list."""
        prev_node, next_node = node.prev, node.next
        prev_node.next = next_node
        next_node.prev = prev_node
    
    def _add_to_front(self, node):
        """Add a node right after the head (most recently used position)."""
        node.prev = self._head
        node.next = self._head.next
        self._head.next.prev = node
        self._head.next = node
    
    def _move_to_front(self, node):
        """Move an existing node to the front (most recently used)."""
        self._remove(node)
        self._add_to_front(node)
    
    def _pop_tail(self):
        """Remove and return the least recently used node."""
        lru_node = self._tail.prev
        self._remove(lru_node)
        return lru_node
    
    def get(self, key: int) -> int:
        """Get the value for a key, marking it as recently used."""
        if key not in self.cache:
            return -1
        node = self.cache[key]
        self._move_to_front(node)
        return node.value
    
    def put(self, key: int, value: int) -> None:
        """Add or update a key-value pair."""
        if key in self.cache:
            node = self.cache[key]
            node.value = value
            self._move_to_front(node)
        else:
            new_node = self._Node(key, value)
            self.cache[key] = new_node
            self._add_to_front(new_node)
            
            if len(self.cache) > self.capacity:
                lru_node = self._pop_tail()
                del self.cache[lru_node.key]

# Test the LRU Cache
lru = LRUCache(2)
lru.put(1, 1)
lru.put(2, 2)
print(f"Get 1: {lru.get(1)}")  # 1
lru.put(3, 3)  # Evicts key 2
print(f"Get 2: {lru.get(2)}")  # -1
lru.put(4, 4)  # Evicts key 1
print(f"Get 1: {lru.get(1)}")  # -1
print(f"Get 3: {lru.get(3)}")  # 3
print(f"Get 4: {lru.get(4)}")  # 4

### Problem 2: Serialize and Deserialize Binary Tree (LeetCode #297)

Implement an encoder/decoder for a binary tree.

Design an algorithm to serialize a binary tree to a single string and deserialize the string back to the original tree structure.

In [ ]:
from collections import deque

class TreeNode:
    """Binary tree node with __slots__ for memory efficiency."""
    __slots__ = ('val', 'left', 'right')
    
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

class Codec:
    """Codec for serializing and deserializing binary trees."""
    
    def serialize(self, root: TreeNode) -> str:
        """Encode a tree to a single string using level-order traversal."""
        if not root:
            return "#"
        
        result = []
        queue = deque([root])
        
        while queue:
            node = queue.popleft()
            if node:
                result.append(str(node.val))
                queue.append(node.left)
                queue.append(node.right)
            else:
                result.append("#")
        
        # Remove trailing null markers
        while result and result[-1] == "#":
            result.pop()
        
        return ",".join(result)
    
    def deserialize(self, data: str) -> TreeNode:
        """Decode a string to a binary tree."""
        if data == "#":
            return None
        
        values = data.split(",")
        root = TreeNode(int(values[0]))
        queue = deque([root])
        i = 1
        
        while queue and i < len(values):
            node = queue.popleft()
            
            # Left child
            if i < len(values) and values[i] != "#":
                node.left = TreeNode(int(values[i]))
                queue.append(node.left)
            i += 1
            
            # Right child
            if i < len(values) and values[i] != "#":
                node.right = TreeNode(int(values[i]))
                queue.append(node.right)
            i += 1
        
        return root

# Helper function to print tree
def print_tree(root, level=0):
    if root:
        print("  " * level + str(root.val))
        print_tree(root.left, level + 1)
        print_tree(root.right, level + 1)

# Test the Codec
codec = Codec()
tree = TreeNode(1)
tree.left = TreeNode(2)
tree.right = TreeNode(3)
tree.right.left = TreeNode(4)
tree.right.right = TreeNode(5)

serialized = codec.serialize(tree)
print(f"Serialized: {serialized}")

deserialized = codec.deserialize(serialized)
print(f"Deserialized tree:")
print_tree(deserialized)

### Problem 3: Flatten Nested List Iterator (LeetCode #341)

Given a nested list of integers, implement an iterator to flatten it.

Each element is either an integer or a nested list. Implement the `NestedIterator` class:
- `next()` returns the next integer
- `hasNext()` returns whether there is a next integer

In [ ]:
class NestedInteger:
    """NestedInteger interface for LeetCode #341."""
    def __init__(self, value):
        if isinstance(value, int):
            self._integer = value
            self._list = None
        else:
            self._integer = None
            self._list = value
    
    def isInteger(self):
        return self._integer is not None
    
    def getInteger(self):
        return self._integer
    
    def getList(self):
        return self._list

class NestedIterator:
    """Iterator to flatten nested lists."""
    
    def __init__(self, nestedList: list):
        self._stack = []
        # Push in reverse order so we can pop from the end
        for ni in reversed(nestedList):
            self._stack.append(ni)
    
    def next(self) -> int:
        """Return the next integer."""
        if self.hasNext():
            return self._stack.pop().getInteger()
        return None
    
    def hasNext(self) -> bool:
        """Check if there is a next integer."""
        while self._stack:
            top = self._stack[-1]
            if top.isInteger():
                return True
            # Pop the nested list and push its elements
            nested = self._stack.pop()
            for ni in reversed(nested.getList()):
                self._stack.append(ni)
        return False

# Test the NestedIterator
nested_list = [
    NestedInteger([NestedInteger(1), NestedInteger(1)]),
    NestedInteger(2),
    NestedInteger([NestedInteger(1), NestedInteger(1)])
]

iterator = NestedIterator(nested_list)
result = []
while iterator.hasNext():
    result.append(iterator.next())
print(f"Flattened: {result}")  # [1, 1, 2, 1, 1]

### Problem 4: Design Add and Search Words (LeetCode #211)

Design a data structure that supports adding new words and searching for words.

Implement the `WordDictionary` class:
- `void addWord(word)` Adds word to the data structure
- `bool search(word)` Returns true if the word is in the data structure

The search word may contain dots '.' where a dot means a single letter.

In [ ]:
class WordDictionary:
    """Word dictionary using a Trie (prefix tree) with wildcard search support."""
    
    class _TrieNode:
        __slots__ = ('children', 'is_end')
        
        def __init__(self):
            self.children = {}
            self.is_end = False
    
    def __init__(self):
        self._root = self._TrieNode()
    
    def addWord(self, word: str) -> None:
        """Add a word to the dictionary."""
        node = self._root
        for char in word:
            if char not in node.children:
                node.children[char] = self._TrieNode()
            node = node.children[char]
        node.is_end = True
    
    def search(self, word: str) -> bool:
        """Search for a word, supporting '.' as wildcard."""
        return self._search_helper(word, 0, self._root)
    
    def _search_helper(self, word: str, index: int, node: '_TrieNode') -> bool:
        """Recursive helper for search with wildcards."""
        if index == len(word):
            return node.is_end
        
        char = word[index]
        if char == '.':
            # Wildcard: try all children
            for child in node.children.values():
                if self._search_helper(word, index + 1, child):
                    return True
            return False
        else:
            if char not in node.children:
                return False
            return self._search_helper(word, index + 1, node.children[char])

# Test the WordDictionary
word_dict = WordDictionary()
word_dict.addWord('bad')
word_dict.addWord('dad');
word_dict.addWord('mad')

print(f"Search 'pad': {word_dict.search('pad')}")  # False
print(f"Search 'bad': {word_dict.search('bad')}")  # True
print(f"Search '.ad': {word_dict.search('.ad')}")  # True
print(f"Search 'b..': {word_dict.search('b..')}")  # True
print(f"Search 'b...': {word_dict.search('b...')}")  # False

## Summary

In this notebook, we covered intermediate OOP concepts in Python:

1. **Abstract Classes**: Using ABC and @abstractmethod to define contracts
2. **Interfaces**: Simulating interfaces with abstract classes
3. **Composition vs Aggregation**: Strong vs weak ownership relationships
4. **@property**: Pythonic getters, setters, and deleters
5. **Name Mangling**: Preventing attribute name collisions
6. **__slots__**: Memory optimization and attribute restriction
7. **Magic Methods**: Customizing Python's built-in behavior
8. **Named Tuples vs Data Classes**: Lightweight data containers
9. **Nested Classes**: Organizing related classes
10. **Practical Applications**: Solving LeetCode problems with OOP

These concepts build upon the basics and prepare you for advanced OOP patterns.